# Extended Data Figure 2 — Molecular preprocessing

The retained cells cover panels b–d and supporting protein-panel analyses. The SNP-selection diagram in panel a is not generated here.


### Extended Data Figure 2b — Selected SNP genomic coverage

Maps the prediction SNPs to cytobands and compares their coverage with known CHD and ischemic stroke GWAS regions.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# ======================================================
# Matplotlib settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

# ======================================================
# Config
# ======================================================

VARIANT_CSV = "CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "gwas-catalog-download-associations-v1.0-full.tsv"

OUTDIR = "figs2_final/snp_cytoband_coverage_preprocessing_onerow"
os.makedirs(OUTDIR, exist_ok=True)

CYTOBAND_FILE = os.path.join(OUTDIR, "hg38_cytoBand.txt.gz")

OUTFIG = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_coverage_known_GWAS_loci_onerow.pdf"
)

OUTSVG = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_coverage_known_GWAS_loci_onerow.svg"
)

OUT_LEGEND_PDF = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_coverage_legend.pdf"
)

OUT_LEGEND_SVG = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_coverage_legend.svg"
)

OUT_COLORBAR = os.path.join(
    OUTDIR,
    "prediction_SNP_density_colorbar.pdf"
)

OUT_COVERAGE_TABLE = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_coverage_known_GWAS_loci_summary.csv"
)

OUT_KNOWN_GWAS_TABLE = os.path.join(
    OUTDIR,
    "selected_known_GWAS_hits_with_prediction_SNP_coverage.csv"
)

OUT_PRED_SNP_BAND_TABLE = os.path.join(
    OUTDIR,
    "prediction_SNP_cytoband_counts.csv"
)

OUT_PRED_SNP_WITH_BAND = os.path.join(
    OUTDIR,
    "prediction_SNPs_with_cytoband.csv"
)

CHR_ORDER = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

# ======================================================
# Manual selected studies
# ======================================================

MANUAL_SELECTED_STUDIES = [
    # istroke
    {"endpoint": "istroke", "FIRST AUTHOR": "Malik R",      "DATE": "2018-03-12", "JOURNAL": "Nat Genet", "PUBMEDID": "29531354"},
    {"endpoint": "istroke", "FIRST AUTHOR": "Mishra A",     "DATE": "2022-09-30", "JOURNAL": "Nature",    "PUBMEDID": "36180795"},
    {"endpoint": "istroke", "FIRST AUTHOR": "Soderholm M",  "DATE": "2019-02-22", "JOURNAL": "Neurology", "PUBMEDID": "30796134"},

    # CHD
    {"endpoint": "CHD", "FIRST AUTHOR": "Aragam KG",      "DATE": "2022-12-06", "JOURNAL": "Nat Genet", "PUBMEDID": "36474045"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Tcheandjieu C",  "DATE": "2022-08-01", "JOURNAL": "Nat Med",   "PUBMEDID": "35915156"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Koyama S",       "DATE": "2020-10-05", "JOURNAL": "Nat Genet", "PUBMEDID": "33020668"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Lu X",           "DATE": "2012-07-01", "JOURNAL": "Nat Genet", "PUBMEDID": "22751097"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Nikpay M",       "DATE": "2015-09-07", "JOURNAL": "Nat Genet", "PUBMEDID": "26343387"},
]

# ======================================================
# disease / trait keywords
# ======================================================

CHD_PATTERNS = [
    r"\bcoronary heart disease\b",
    r"\bcoronary artery disease\b",
    r"\bischemic heart disease\b",
    r"\bmyocardial infarction\b",
    r"\bangina\b",
    r"\bunstable angina\b",
    r"\bacute coronary syndrome\b",
    r"\batherosclerosis\b",
    r"\bcoronary atherosclerosis\b",
    r"\bcoronary stenosis\b",
    r"\bchd\b",
    r"\bcad\b",
]

ISTROKE_PATTERNS = [
    r"\bischemic stroke\b",
    r"\bischaemic stroke\b",
    r"\bstroke\b",
    r"\bcerebral infarction\b",
    r"\bbrain infarction\b",
    r"\blacunar stroke\b",
    r"\bcardioembolic stroke\b",
    r"\blarge artery stroke\b",
    r"\bsmall vessel stroke\b",
    r"\bsmall artery occlusion\b",
    r"\btransient ischemic attack\b",
    r"\btransient ischaemic attack\b",
    r"\btia\b",
]

# ======================================================
# Plot config
# ======================================================


FIGSIZE_MM = (200, 70)
FIGSIZE = (FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4)


CHROM_WIDTH = 0.16
SNP_TRACK_WIDTH = 0.055
SNP_TRACK_GAP = 0.035

COL_GAP = 0.43

TOP_PAD_MB = 14
BOTTOM_PAD_MB = 18

COLOR_CHROM = "#F2F2F2"
COLOR_CHROM_EDGE = "#9E9E9E"

COLOR_KNOWN_CHD = "#D55E5E"
COLOR_KNOWN_ISTROKE = "#5B8FD9"
COLOR_KNOWN_BOTH = "#9B59B6"

COLOR_COVERED = "#111111"

SNP_CMAP = LinearSegmentedColormap.from_list(
    "snp_density_grey",
    ["#EEEEEE", "#111111"]
)

# ======================================================
# Helper functions
# ======================================================

def download_cytoband_if_needed(path_out):
    if os.path.exists(path_out):
        return

    url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cytoBand.txt.gz"
    print("Downloading cytoband file from:", url)
    urllib.request.urlretrieve(url, path_out)
    print("Saved:", path_out)


def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_date_to_yyyy_mm_dd(x):
    dt = pd.to_datetime(x, errors="coerce")
    if pd.isna(dt):
        return normalize_text(x)
    return dt.strftime("%Y-%m-%d")


def normalize_gwas_chr(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s.upper() == "X":
        return "chrX"
    if s.upper() == "Y":
        return "chrY"

    if re.fullmatch(r"\d+", s):
        val = int(s)
        if val == 23:
            return "chrX"
        if val == 24:
            return "chrY"
        return f"chr{val}"

    return np.nan


def parse_variant(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    return np.nan, np.nan


def chrom_sort_key(chrom):
    x = str(chrom).replace("chr", "")
    if x == "X":
        return 23
    if x == "Y":
        return 24
    return int(x)


def build_regex(patterns):
    return re.compile("|".join(patterns), flags=re.IGNORECASE)


def trait_match(series, regex_obj):
    return series.fillna("").astype(str).str.contains(regex_obj, regex=True, na=False)


def load_cytobands(path_in):
    bands = pd.read_csv(
        path_in,
        sep="\t",
        compression="gzip",
        header=None,
        names=["chrom", "start", "end", "band_name", "gieStain"]
    )

    bands = bands[bands["chrom"].isin(CHR_ORDER)].copy()
    bands = bands.sort_values(["chrom", "start"]).reset_index(drop=True)

    bands["band_id"] = (
        bands["chrom"].astype(str) + ":" +
        bands["band_name"].astype(str) + ":" +
        bands["start"].astype(str) + "-" +
        bands["end"].astype(str)
    )

    bands["cytoband"] = (
        bands["chrom"].str.replace("chr", "", regex=False) +
        bands["band_name"].astype(str)
    )

    bands["start_mb"] = bands["start"] / 1e6
    bands["end_mb"] = bands["end"] / 1e6
    bands["len_bp"] = bands["end"] - bands["start"]
    bands["len_mb"] = bands["len_bp"] / 1e6

    return bands


def map_positions_to_bands(df, bands_df, chrom_col="chr", pos_col="pos"):
    if df.empty:
        return df.copy()

    out = []

    for chrom, sub in df.groupby(chrom_col):
        band_sub = bands_df[bands_df["chrom"] == chrom].sort_values("start").copy()

        if band_sub.empty:
            tmp = sub.copy()
            tmp["band_id"] = np.nan
            tmp["band_name"] = np.nan
            tmp["cytoband"] = np.nan
            out.append(tmp)
            continue

        starts = band_sub["start"].to_numpy()
        ends = band_sub["end"].to_numpy()
        band_ids = band_sub["band_id"].to_numpy()
        band_names = band_sub["band_name"].to_numpy()
        cytobands = band_sub["cytoband"].to_numpy()

        pos = sub[pos_col].to_numpy()
        idx = np.searchsorted(starts, pos, side="right") - 1

        band_id_arr = np.array([np.nan] * len(sub), dtype=object)
        band_name_arr = np.array([np.nan] * len(sub), dtype=object)
        cytoband_arr = np.array([np.nan] * len(sub), dtype=object)

        valid = idx >= 0
        valid_idx = idx[valid]
        valid_pos = pos[valid]
        inside = valid_pos < ends[valid_idx]

        pos_valid_indices = np.where(valid)[0][inside]
        band_valid_indices = valid_idx[inside]

        band_id_arr[pos_valid_indices] = band_ids[band_valid_indices]
        band_name_arr[pos_valid_indices] = band_names[band_valid_indices]
        cytoband_arr[pos_valid_indices] = cytobands[band_valid_indices]

        tmp = sub.copy()
        tmp["band_id"] = band_id_arr
        tmp["band_name"] = band_name_arr
        tmp["cytoband"] = cytoband_arr

        out.append(tmp)

    return pd.concat(out, ignore_index=True)


def apply_manual_study_flags_all_hits(gwas_df, manual_studies):
    gwas_df = gwas_df.copy()

    gwas_df["is_chd_related"] = False
    gwas_df["is_istroke_related"] = False
    gwas_df["manual_study_selected"] = False

    for col in ["FIRST AUTHOR", "JOURNAL", "PUBMEDID"]:
        if col in gwas_df.columns:
            gwas_df[f"__{col}"] = gwas_df[col].apply(normalize_text)
        else:
            gwas_df[f"__{col}"] = ""

    if "DATE" not in gwas_df.columns:
        raise ValueError("GWAS catalog does not contain DATE column.")

    gwas_df["__DATE"] = gwas_df["DATE"].apply(normalize_date_to_yyyy_mm_dd)

    trait_series = gwas_df["DISEASE/TRAIT"].fillna("").astype(str)
    study_series = gwas_df.get(
        "STUDY",
        pd.Series([""] * len(gwas_df), index=gwas_df.index)
    ).fillna("").astype(str)

    chd_regex = build_regex(CHD_PATTERNS)
    istroke_regex = build_regex(ISTROKE_PATTERNS)

    chd_trait_mask = (
        trait_match(trait_series, chd_regex) |
        trait_match(study_series, chd_regex)
    )

    istroke_trait_mask = (
        trait_match(trait_series, istroke_regex) |
        trait_match(study_series, istroke_regex)
    )

    for item in manual_studies:
        endpoint = item["endpoint"]
        author = normalize_text(item["FIRST AUTHOR"])
        date = normalize_date_to_yyyy_mm_dd(item["DATE"])
        journal = normalize_text(item["JOURNAL"])
        pubmedid = normalize_text(item["PUBMEDID"])

        study_mask = (
            (gwas_df["__FIRST AUTHOR"] == author) &
            (gwas_df["__DATE"] == date) &
            (gwas_df["__JOURNAL"] == journal) &
            (gwas_df["__PUBMEDID"] == pubmedid)
        )

        if endpoint == "CHD":
            final_mask = study_mask & chd_trait_mask
            gwas_df.loc[final_mask, "is_chd_related"] = True

        elif endpoint == "istroke":
            final_mask = study_mask & istroke_trait_mask
            gwas_df.loc[final_mask, "is_istroke_related"] = True

        else:
            raise ValueError(f"Unknown endpoint: {endpoint}")

        gwas_df.loc[study_mask, "manual_study_selected"] = True

        print(
            f"[Manual study] {endpoint} | {author} {date} {journal} PMID={pubmedid} | "
            f"all hits in study = {int(study_mask.sum())} | "
            f"endpoint-matched hits = {int(final_mask.sum())}"
        )

    helper_cols = [c for c in gwas_df.columns if c.startswith("__")]
    gwas_df = gwas_df.drop(columns=helper_cols)

    return gwas_df


def known_region_type(row):
    if row["known_chd_hits"] > 0 and row["known_istroke_hits"] > 0:
        return "both"
    if row["known_chd_hits"] > 0:
        return "chd"
    if row["known_istroke_hits"] > 0:
        return "istroke"
    return "none"


def known_region_color(region_type):
    if region_type == "chd":
        return COLOR_KNOWN_CHD
    if region_type == "istroke":
        return COLOR_KNOWN_ISTROKE
    if region_type == "both":
        return COLOR_KNOWN_BOTH
    return COLOR_CHROM


def known_region_fill(region_type):
    if region_type == "chd":
        return "#F5CACA"
    if region_type == "istroke":
        return "#CFE0F6"
    if region_type == "both":
        return "#E6D4F0"
    return COLOR_CHROM


# ======================================================
# Main
# ======================================================

download_cytoband_if_needed(CYTOBAND_FILE)
bands = load_cytobands(CYTOBAND_FILE)

chrom_sizes = (
    bands.groupby("chrom", as_index=False)["end_mb"]
         .max()
         .rename(columns={"end_mb": "chrom_len_mb"})
)
chrom_len_dict = dict(zip(chrom_sizes["chrom"], chrom_sizes["chrom_len_mb"]))

# ======================================================
# 1. Load prediction SNPs
# ======================================================

variant = pd.read_csv(VARIANT_CSV)

if "variant_id" not in variant.columns:
    raise ValueError("VARIANT_CSV must contain 'variant_id' column.")

if "SNP" not in variant.columns:
    raise ValueError("VARIANT_CSV must contain 'SNP' column.")

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant(x))
)

variant = variant.dropna(subset=["chr", "pos"]).copy()
variant["pos"] = variant["pos"].astype(int)
variant = variant[variant["chr"].isin(CHR_ORDER)].copy()

pred_snp_band = map_positions_to_bands(
    variant,
    bands,
    chrom_col="chr",
    pos_col="pos"
)

pred_snp_band = pred_snp_band.dropna(subset=["band_id"]).copy()

pred_band_counts = (
    pred_snp_band
    .groupby("band_id", as_index=False)
    .agg(
        n_prediction_snps=("SNP", "nunique"),
        n_prediction_rows=("SNP", "size")
    )
)

pred_snp_band.to_csv(OUT_PRED_SNP_WITH_BAND, index=False)

# ======================================================
# 2. Load selected known GWAS hits
# ======================================================

gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)
gwas.columns = gwas.columns.str.strip()

if "CHR_ID" not in gwas.columns or "CHR_POS" not in gwas.columns:
    raise ValueError("GWAS catalog must contain CHR_ID and CHR_POS columns.")

gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")
gwas = gwas.dropna(subset=["CHR_ID", "CHR_POS"]).copy()

gwas["chr"] = gwas["CHR_ID"].apply(normalize_gwas_chr)
gwas = gwas[gwas["chr"].isin(CHR_ORDER)].copy()
gwas["pos"] = gwas["CHR_POS"].astype(int)

print("\nGWAS chromosome counts after normalization:")
print(gwas["chr"].value_counts().reindex(CHR_ORDER).fillna(0).astype(int).to_string())

gwas_flagged = apply_manual_study_flags_all_hits(
    gwas_df=gwas,
    manual_studies=MANUAL_SELECTED_STUDIES
)

known_hits = gwas_flagged[
    gwas_flagged["is_chd_related"] |
    gwas_flagged["is_istroke_related"]
].copy()

known_hits_mapped = map_positions_to_bands(
    known_hits,
    bands,
    chrom_col="chr",
    pos_col="pos"
)

known_hits_mapped = known_hits_mapped.dropna(subset=["band_id"]).copy()

# ======================================================
# 3. Cytoband-level table
# ======================================================

band_table = bands[
    [
        "band_id",
        "chrom",
        "band_name",
        "cytoband",
        "start",
        "end",
        "start_mb",
        "end_mb",
        "len_bp",
        "len_mb",
    ]
].copy()

band_table = band_table.merge(
    pred_band_counts,
    on="band_id",
    how="left"
)

band_table["n_prediction_snps"] = band_table["n_prediction_snps"].fillna(0).astype(int)
band_table["n_prediction_rows"] = band_table["n_prediction_rows"].fillna(0).astype(int)
band_table["has_prediction_snp"] = band_table["n_prediction_snps"] > 0

known_all_counts = (
    known_hits_mapped
    .groupby("band_id", as_index=False)
    .size()
    .rename(columns={"size": "known_hits_total"})
)

known_chd_counts = (
    known_hits_mapped[known_hits_mapped["is_chd_related"]]
    .groupby("band_id", as_index=False)
    .size()
    .rename(columns={"size": "known_chd_hits"})
)

known_istroke_counts = (
    known_hits_mapped[known_hits_mapped["is_istroke_related"]]
    .groupby("band_id", as_index=False)
    .size()
    .rename(columns={"size": "known_istroke_hits"})
)

band_table = band_table.merge(known_all_counts, on="band_id", how="left")
band_table = band_table.merge(known_chd_counts, on="band_id", how="left")
band_table = band_table.merge(known_istroke_counts, on="band_id", how="left")

for col in ["known_hits_total", "known_chd_hits", "known_istroke_hits"]:
    band_table[col] = band_table[col].fillna(0).astype(int)

band_table["known_region_type"] = band_table.apply(known_region_type, axis=1)
band_table["known_region_has_prediction_snp"] = (
    (band_table["known_hits_total"] > 0) &
    (band_table["has_prediction_snp"])
)

band_table.to_csv(OUT_PRED_SNP_BAND_TABLE, index=False)

# ======================================================
# 4. Coverage of known GWAS loci
# ======================================================

known_hits_mapped = known_hits_mapped.merge(
    band_table[["band_id", "has_prediction_snp", "n_prediction_snps"]],
    on="band_id",
    how="left"
)

known_hits_mapped["covered_by_prediction_snp_cytoband"] = (
    known_hits_mapped["has_prediction_snp"].fillna(False)
)

known_hits_mapped.to_csv(OUT_KNOWN_GWAS_TABLE, index=False)

n_known_hits = len(known_hits_mapped)
n_known_hits_covered = int(known_hits_mapped["covered_by_prediction_snp_cytoband"].sum())

coverage_ratio = (
    n_known_hits_covered / n_known_hits
    if n_known_hits > 0 else np.nan
)

summary_rows = []

for endpoint, col in [
    ("CHD", "is_chd_related"),
    ("istroke", "is_istroke_related"),
]:
    sub = known_hits_mapped[known_hits_mapped[col]].copy()
    n = len(sub)
    n_cov = int(sub["covered_by_prediction_snp_cytoband"].sum())

    summary_rows.append({
        "endpoint": endpoint,
        "known_GWAS_hits_mapped_to_cytoband": n,
        "known_GWAS_hits_covered_by_prediction_SNP_cytoband": n_cov,
        "coverage_ratio": n_cov / n if n > 0 else np.nan,
    })

summary_rows.append({
    "endpoint": "CHD_or_istroke",
    "known_GWAS_hits_mapped_to_cytoband": n_known_hits,
    "known_GWAS_hits_covered_by_prediction_SNP_cytoband": n_known_hits_covered,
    "coverage_ratio": coverage_ratio,
})

coverage_summary = pd.DataFrame(summary_rows)
coverage_summary.to_csv(OUT_COVERAGE_TABLE, index=False)

print("\n==============================")
print("Prediction SNP cytoband coverage of known GWAS loci")
print("==============================")
print(coverage_summary.to_string(index=False))

print(
    f"\nMain coverage number: "
    f"{n_known_hits_covered:,} / {n_known_hits:,} = {coverage_ratio:.2%}"
)

# ======================================================
# 5. One-row chromosome plot
# ======================================================

plot_chroms = [c for c in CHR_ORDER if c in chrom_len_dict]
max_chr_len = max([chrom_len_dict[c] for c in plot_chroms])

x_positions = {
    chrom: i * COL_GAP
    for i, chrom in enumerate(plot_chroms)
}

# SNP density normalization
snp_values = band_table["n_prediction_snps"].to_numpy()
snp_values_nonzero = snp_values[snp_values > 0]

if len(snp_values_nonzero) > 0:
    vmax = np.quantile(snp_values_nonzero, 0.99)
    if vmax <= 0:
        vmax = snp_values_nonzero.max()
else:
    vmax = 1

norm = Normalize(vmin=0, vmax=vmax)

fig, ax = plt.subplots(figsize=FIGSIZE)

for chrom in plot_chroms:

    x0 = x_positions[chrom]
    y_base = 0

    sub = band_table[band_table["chrom"] == chrom].sort_values("start").copy()

    chrom_x = x0
    snp_track_x = chrom_x + CHROM_WIDTH + SNP_TRACK_GAP

    chrom_len = chrom_len_dict[chrom]

    # -----------------------------
    # chromosome band body
    # -----------------------------
    for _, row in sub.iterrows():
        y = y_base + row["start_mb"]
        h = row["len_mb"]

        region_type = row["known_region_type"]
        face = known_region_fill(region_type)

        ax.add_patch(
            Rectangle(
                (chrom_x, y),
                CHROM_WIDTH,
                h,
                facecolor=face,
                edgecolor="white",
                linewidth=0.08,
                zorder=1
            )
        )

        # known GWAS cytoband outline
        if row["known_hits_total"] > 0:
            edge_color = known_region_color(region_type)

            ax.add_patch(
                Rectangle(
                    (chrom_x - 0.010, y),
                    CHROM_WIDTH + 0.020,
                    h,
                    facecolor="none",
                    edgecolor=edge_color,
                    linewidth=0.50,
                    zorder=4
                )
            )


    # chromosome outline
    ax.add_patch(
        Rectangle(
            (chrom_x, y_base),
            CHROM_WIDTH,
            chrom_len,
            facecolor="none",
            edgecolor=COLOR_CHROM_EDGE,
            linewidth=0.55,
            zorder=5
        )
    )

    # -----------------------------
    # prediction SNP density track
    # -----------------------------
    for _, row in sub.iterrows():
        y = y_base + row["start_mb"]
        h = row["len_mb"]

        n_snp = row["n_prediction_snps"]

        if n_snp > 0:
            color = SNP_CMAP(norm(n_snp))
        else:
            color = "#FFFFFF"

        ax.add_patch(
            Rectangle(
                (snp_track_x, y),
                SNP_TRACK_WIDTH,
                h,
                facecolor=color,
                edgecolor="white",
                linewidth=0.05,
                zorder=2
            )
        )

    ax.add_patch(
        Rectangle(
            (snp_track_x, y_base),
            SNP_TRACK_WIDTH,
            chrom_len,
            facecolor="none",
            edgecolor=COLOR_CHROM_EDGE,
            linewidth=0.45,
            zorder=5
        )
    )

    # chromosome label
    ax.text(
        chrom_x + CHROM_WIDTH / 2,
        y_base - 7,
        chrom.replace("chr", ""),
        ha="center",
        va="bottom",
        fontsize=6.5
    )

# ======================================================
# Coverage number annotation
# ======================================================

coverage_text = (
    f"Known GWAS loci covered by prediction SNP cytobands: "
    f"{n_known_hits_covered:,}/{n_known_hits:,} ({coverage_ratio:.1%})"
)

ax.text(
    0.01,
    0.985,
    coverage_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=7.5
)

# ======================================================
# Style
# ======================================================

x_max = (
    (len(plot_chroms) - 1) * COL_GAP
    + CHROM_WIDTH
    + SNP_TRACK_GAP
    + SNP_TRACK_WIDTH
    + 0.30
)

y_min = -TOP_PAD_MB
y_max = max_chr_len + BOTTOM_PAD_MB

ax.set_xlim(-0.20, x_max)
ax.set_ylim(y_max, y_min)

ax.set_xticks([])
ax.set_yticks([])

for spine in ["top", "right", "left", "bottom"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True, bbox_inches="tight")
plt.savefig(OUTSVG, dpi=300, transparent=True, bbox_inches="tight")
plt.close()

print("\nSaved main figure:")
print(OUTFIG)
print(OUTSVG)

# ======================================================
# 6. Separate legend
# ======================================================

legend_handles = [
    Patch(facecolor="#F5CACA", edgecolor=COLOR_KNOWN_CHD, label="Known CHD cytoband"),
    Patch(facecolor="#CFE0F6", edgecolor=COLOR_KNOWN_ISTROKE, label="Known iStroke cytoband"),
    Patch(facecolor="#E6D4F0", edgecolor=COLOR_KNOWN_BOTH, label="Shared known cytoband"),
    Patch(facecolor=COLOR_COVERED, edgecolor="none", label="Covered by prediction SNPs"),
    Patch(facecolor="#FFFFFF", edgecolor=COLOR_CHROM_EDGE, label="Prediction SNP density track"),
]

fig_leg, ax_leg = plt.subplots(figsize=(3.2, 1.4))
ax_leg.axis("off")

ax_leg.legend(
    handles=legend_handles,
    frameon=False,
    fontsize=8,
    loc="center",
    ncol=1
)

plt.tight_layout()
plt.savefig(OUT_LEGEND_PDF, dpi=300, transparent=True, bbox_inches="tight")
plt.savefig(OUT_LEGEND_SVG, dpi=300, transparent=True, bbox_inches="tight")
plt.close(fig_leg)

print("\nSaved separate legend:")
print(OUT_LEGEND_PDF)
print(OUT_LEGEND_SVG)

# ======================================================
# 7. Separate SNP density colorbar
# ======================================================

fig_cb, ax_cb = plt.subplots(figsize=(1.0, 2.6))

sm = ScalarMappable(norm=norm, cmap=SNP_CMAP)
sm.set_array([])

cbar = plt.colorbar(
    sm,
    cax=ax_cb,
    orientation="vertical"
)

cb_ticks = np.linspace(0, vmax, 3)
cbar.set_ticks(cb_ticks)
cbar.set_ticklabels([f"{x:.0f}" for x in cb_ticks])

cbar.set_label("Prediction SNPs per cytoband", fontsize=8)
cbar.ax.tick_params(labelsize=7)

plt.savefig(
    OUT_COLORBAR,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.close(fig_cb)

print("\nSaved SNP density colorbar:")
print(OUT_COLORBAR)

# ======================================================
# Useful prints
# ======================================================

print("\nTop cytobands by prediction SNP count:")
print(
    band_table.sort_values("n_prediction_snps", ascending=False)
    [
        [
            "chrom",
            "band_name",
            "cytoband",
            "n_prediction_snps",
            "known_hits_total",
            "known_chd_hits",
            "known_istroke_hits",
            "has_prediction_snp",
        ]
    ]
    .head(30)
    .to_string(index=False)
)

print("\nKnown GWAS cytobands not covered by prediction SNPs:")
not_covered = band_table[
    (band_table["known_hits_total"] > 0) &
    (~band_table["has_prediction_snp"])
].copy()

if not_covered.empty:
    print("None. All known GWAS cytobands are covered by at least one prediction SNP.")
else:
    print(
        not_covered[
            [
                "chrom",
                "band_name",
                "cytoband",
                "known_hits_total",
                "known_chd_hits",
                "known_istroke_hits",
                "n_prediction_snps",
            ]
        ].to_string(index=False)
    )

### Supports Extended Data Figure 2b — Genomic coverage table

Exports the coverage summary from the preceding cytoband analysis as LaTeX.


In [ ]:
from pathlib import Path
import pandas as pd


OUT_COVERAGE_TEX = (
    Path(OUTDIR)
    / "prediction_SNP_cytoband_coverage_table.tex"
)


required_columns = [
    "endpoint",
    "known_GWAS_hits_mapped_to_cytoband",
    "known_GWAS_hits_covered_by_prediction_SNP_cytoband",
    "coverage_ratio",
]

missing_columns = [
    column
    for column in required_columns
    if column not in coverage_summary.columns
]

if missing_columns:
    raise ValueError(
        "coverage_summary is missing columns:"
        + ", ".join(missing_columns)
    )


endpoint_labels = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "CHD_or_istroke": "CHD or ischemic stroke",
}


table_rows = []

for _, row in coverage_summary.iterrows():

    endpoint = endpoint_labels.get(
        str(row["endpoint"]),
        str(row["endpoint"])
    )

    n_known = int(
        row["known_GWAS_hits_mapped_to_cytoband"]
    )

    n_covered = int(
        row[
            "known_GWAS_hits_covered_by_prediction_SNP_cytoband"
        ]
    )

    ratio = row["coverage_ratio"]

    if pd.isna(ratio):
        ratio_text = "NA"
    else:
        ratio_text = f"{float(ratio) * 100:.1f}\\%"

    table_rows.append(
        f"        {endpoint} & "
        f"{n_known:,} & "
        f"{n_covered:,} & "
        f"{ratio_text} \\\\"
    )

rows_tex = "\n".join(table_rows)


tex_content = rf"""\begin{{table}}[ht]
    \centering
    \caption{{Coverage of known GWAS loci by cytobands containing prediction SNPs.}}
    \label{{tab:gwas-cytoband-coverage}}
    \begin{{threeparttable}}
        \begin{{tabular}}{{lrrr}}
            \toprule
            Endpoint &
            Known GWAS hits &
            Covered hits &
            Coverage \\
            \midrule
{rows_tex}
            \bottomrule
        \end{{tabular}}
        \begin{{tablenotes}}[flushleft]
            \footnotesize
            \item A known GWAS hit was considered covered when its
            cytoband contained at least one prediction SNP.
        \end{{tablenotes}}
    \end{{threeparttable}}
\end{{table}}
"""


OUT_COVERAGE_TEX.parent.mkdir(
    parents=True,
    exist_ok=True
)

OUT_COVERAGE_TEX.write_text(
    tex_content,
    encoding="utf-8"
)

print("Saved LaTeX table:")
print(OUT_COVERAGE_TEX.resolve())


print("\n" + tex_content)

### Extended Data Figure 2c — Protein normalization comparison

Compares IRN and log1p protein normalization. The implementation selects the five highest-ranked IRN runs and five lowest-ranked log1p runs.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl


# ======================================================
# Global matplotlib settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

IRN_DIR = Path("code_9_15_save/SingleModal_GS_Run_IRN")
NORM_DIR = Path("code_9_8_save/SingleModal_GS_Run_v2")

OUTDIR = Path("IRN_vs_normlog1p_performance")
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_PERFORMANCE_TABLE = OUTDIR / "IRN_best5_normlog1p_worst5_CHD_istroke_ICVD_performance.csv"
OUT_SELECTED_FOLDER_TABLE = OUTDIR / "selected_folders_rank_summary.csv"
OUT_FIG = OUTDIR / "IRN_best5_vs_normlog1p_worst5_barplot.pdf"

ENDPOINTS = ["CHD", "istroke", "ICVD"]

COLOR_IRN = "#861e20"
COLOR_NORM = "#B0B0B0"

FIGSIZE = (3.6, 3.0)


# ======================================================
# Functions
# ======================================================

def read_all_soma_summary(base_dir: Path, method_name: str) -> pd.DataFrame:
    """
    Read all soma*/summary.csv under base_dir.
    Return long-format dataframe.
    """

    rows = []

    soma_dirs = sorted([
        p for p in base_dir.iterdir()
        if p.is_dir() and p.name.startswith("soma")
    ])

    print(f"[INFO] {method_name}: found {len(soma_dirs)} soma folders")

    for folder in soma_dirs:
        summary_file = folder / "summary.csv"

        if not summary_file.exists():
            print(f"[WARN] missing summary.csv: {folder}")
            continue

        try:
            df = pd.read_csv(summary_file)
        except Exception as e:
            print(f"[WARN] failed to read {summary_file}: {e}")
            continue

        required_cols = {"endpoint", "modality", "model", "cindex"}
        if not required_cols.issubset(df.columns):
            print(f"[WARN] bad columns in {summary_file}: {df.columns.tolist()}")
            continue

        df = df.copy()
        df["method"] = method_name
        df["folder"] = folder.name
        df["summary_path"] = str(summary_file)

        rows.append(df)

    if len(rows) == 0:
        raise RuntimeError(f"No valid summary.csv found under {base_dir}")

    out = pd.concat(rows, axis=0, ignore_index=True)
    out["cindex"] = pd.to_numeric(out["cindex"], errors="coerce")

    return out


def rank_folders_by_target_endpoints(
    df: pd.DataFrame,
    endpoints: list[str],
    ascending: bool = False
) -> pd.DataFrame:
    """
    Rank folders by average cindex across selected endpoints.
    ascending=False means best first.
    ascending=True means worst first.
    """

    sub = df[df["endpoint"].isin(endpoints)].copy()

    # keep folders that have all required endpoints
    count_df = (
        sub.groupby("folder")["endpoint"]
        .nunique()
        .reset_index(name="n_endpoints")
    )

    valid_folders = count_df.loc[
        count_df["n_endpoints"] == len(endpoints),
        "folder"
    ]

    sub = sub[sub["folder"].isin(valid_folders)].copy()

    rank_df = (
        sub.groupby("folder", as_index=False)
        .agg(
            mean_cindex=("cindex", "mean"),
            sd_cindex=("cindex", "std"),
            n_endpoint=("endpoint", "nunique")
        )
        .sort_values("mean_cindex", ascending=ascending)
        .reset_index(drop=True)
    )

    rank_df["rank"] = np.arange(1, len(rank_df) + 1)

    return rank_df


def summarize_for_plot(perf_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each method and endpoint, summarize mean and SD across selected 5 folders.
    """

    plot_df = (
        perf_df.groupby(["method", "endpoint"], as_index=False)
        .agg(
            mean_cindex=("cindex", "mean"),
            sd_cindex=("cindex", "std"),
            sem_cindex=("cindex", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
            n=("cindex", "count")
        )
    )

    return plot_df


# ======================================================
# Main
# ======================================================

def main():

    # -----------------------------
    # 1. Read all summaries
    # -----------------------------
    df_irn = read_all_soma_summary(IRN_DIR, "IRN")
    df_norm = read_all_soma_summary(NORM_DIR, "norm+log1p")

    # -----------------------------
    # 2. Rank folders
    # -----------------------------
    irn_rank = rank_folders_by_target_endpoints(
        df_irn,
        endpoints=ENDPOINTS,
        ascending=False
    )

    norm_rank = rank_folders_by_target_endpoints(
        df_norm,
        endpoints=ENDPOINTS,
        ascending=True
    )

    irn_best5 = irn_rank.head(5).copy()
    norm_worst5 = norm_rank.head(5).copy()

    irn_best5["method"] = "IRN"
    irn_best5["selection"] = "best5"

    norm_worst5["method"] = "norm+log1p"
    norm_worst5["selection"] = "worst5"

    selected_rank_df = pd.concat([irn_best5, norm_worst5], axis=0, ignore_index=True)
    selected_rank_df.to_csv(OUT_SELECTED_FOLDER_TABLE, index=False)

    print("\n[INFO] IRN best 5 folders:")
    print(irn_best5[["rank", "folder", "mean_cindex"]])

    print("\n[INFO] norm+log1p worst 5 folders:")
    print(norm_worst5[["rank", "folder", "mean_cindex"]])

    # -----------------------------
    # 3. Extract CHD, istroke, ICVD performance
    # -----------------------------
    irn_perf = df_irn[
        (df_irn["folder"].isin(irn_best5["folder"])) &
        (df_irn["endpoint"].isin(ENDPOINTS))
    ].copy()

    norm_perf = df_norm[
        (df_norm["folder"].isin(norm_worst5["folder"])) &
        (df_norm["endpoint"].isin(ENDPOINTS))
    ].copy()

    perf_df = pd.concat([irn_perf, norm_perf], axis=0, ignore_index=True)

    perf_df = perf_df[
        [
            "method",
            "folder",
            "endpoint",
            "modality",
            "model",
            "cindex",
            "summary_path"
        ]
    ].copy()

    perf_df["endpoint"] = pd.Categorical(
        perf_df["endpoint"],
        categories=ENDPOINTS,
        ordered=True
    )

    perf_df = perf_df.sort_values(["method", "folder", "endpoint"])
    perf_df.to_csv(OUT_PERFORMANCE_TABLE, index=False)

    print(f"\n[INFO] saved performance table:")
    print(OUT_PERFORMANCE_TABLE)

    print(f"\n[INFO] saved selected folder ranking table:")
    print(OUT_SELECTED_FOLDER_TABLE)

    # -----------------------------
    # 4. Summarize for plot
    # -----------------------------
    plot_df = summarize_for_plot(perf_df)

    plot_df["endpoint"] = pd.Categorical(
        plot_df["endpoint"],
        categories=ENDPOINTS,
        ordered=True
    )

    plot_df["method"] = pd.Categorical(
        plot_df["method"],
        categories=["IRN", "norm+log1p"],
        ordered=True
    )

    plot_df = plot_df.sort_values(["endpoint", "method"]).reset_index(drop=True)

    print("\n[INFO] plot summary:")
    print(plot_df)

    # -----------------------------
    # 5. Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=FIGSIZE)

    x = np.arange(len(ENDPOINTS))
    width = 0.32

    method_to_color = {
        "IRN": COLOR_IRN,
        "norm+log1p": COLOR_NORM
    }

    offsets = {
        "IRN": -width / 2,
        "norm+log1p": width / 2
    }

    for method in ["IRN", "norm+log1p"]:
        sub = plot_df[plot_df["method"] == method].copy()
        sub = sub.set_index("endpoint").loc[ENDPOINTS].reset_index()

        ax.bar(
            x + offsets[method],
            sub["mean_cindex"],
            width=width,
            color=method_to_color[method],
            edgecolor="black",
            linewidth=0.6,
            label=method,
            zorder=3
        )

        ax.errorbar(
            x + offsets[method],
            sub["mean_cindex"],
            yerr=sub["sd_cindex"],
            fmt="none",
            ecolor="black",
            elinewidth=0.8,
            capsize=2.5,
            capthick=0.8,
            zorder=4
        )

    ax.set_xticks(x)
    ax.set_xticklabels(["CHD", "Ischemic stroke", "ASCVD"], rotation=0)

    ax.set_ylabel("C-index")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.35, zorder=0)

    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=2,
        handlelength=1.2,
        columnspacing=1.2
    )

    
    y_min = plot_df["mean_cindex"].min() - plot_df["sd_cindex"].fillna(0).max() - 0.03
    y_max = plot_df["mean_cindex"].max() + plot_df["sd_cindex"].fillna(0).max() + 0.03

    y_min = max(0.45, y_min)
    y_max = min(0.85, y_max)

    ax.set_ylim(y_min, y_max)

    plt.tight_layout()
    fig.savefig(OUT_FIG, bbox_inches="tight")
    plt.close(fig)

    print(f"\n[INFO] saved figure:")
    print(OUT_FIG)


if __name__ == "__main__":
    main()

### Supports Extended Data Figure 2d — Protein-panel comparison tables

Collects performance results for all-protein, SVM-selected and published protein panels and writes summary tables.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path


# ======================================================
# CONFIG
# ======================================================

ENDPOINTS = ["CHD", "istroke", "ICVD"]

IRN_ROOT = Path("code_9_15_save/SingleModal_GS_Run_IRN")

SVM_DIR = Path("CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1322/curves")
NCR_DIR = Path("CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1324/curves")

CELL_FILE = "STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv"

OUT_RAW = "FINAL_SELECTION_RAW.csv"
OUT_STATS = "FINAL_SELECTION_STATS.csv"


# ======================================================
# 1. IRN (folder-level)
# ======================================================

def load_irn(root: Path):

    rows = []

    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        if not folder.name.startswith("soma"):
            continue

        f = folder / "summary.csv"
        if not f.exists():
            continue

        df = pd.read_csv(f)

        for _, r in df.iterrows():
            if r["endpoint"] not in ENDPOINTS:
                continue

            rows.append({
                "selection": "IRN_all",
                "endpoint": r["endpoint"],
                "mean": float(r["cindex"]),
                "std": np.nan,
                "source": str(f)
            })

    return pd.DataFrame(rows)


# ======================================================
# 2. SVM / NCR (fold-level)
# ======================================================

def load_curve(curve_dir: Path, name: str):

    rows = []
    pattern = re.compile(r"(CHD|stroke|istroke|ICVD)_.*fold(\d+).*\.csv")

    for f in curve_dir.glob("*_train_curve.csv"):

        m = pattern.search(f.name)
        if not m:
            continue

        endpoint = m.group(1)

        if endpoint not in ENDPOINTS:
            continue

        df = pd.read_csv(f)

        if "valid_ci" not in df.columns:
            continue

        rows.append({
            "selection": name,
            "endpoint": endpoint,
            "mean": float(df["valid_ci"].max()),
            "std": np.nan,
            "source": str(f)
        })

    return pd.DataFrame(rows)


# ======================================================
# 3. CELL SELECTED (CV-level TRUE SOURCE)
# ======================================================

def load_cell(cell_file: str):

    df = pd.read_csv(cell_file)

    df = df[df["modality"] == "soma"].copy()
    df = df[df["model_tag"] == "MLP"].copy()
    df = df[df["endpoint"].isin(ENDPOINTS)].copy()

    # unify names
    df = df.rename(columns={
        "mean_valid_ci": "mean",
        "std_valid_ci": "std"
    })

    df["selection"] = "CELL_selected"
    df["source"] = cell_file

    return df[["selection", "endpoint", "mean", "std", "source"]]


# ======================================================
# 4. MERGE
# ======================================================

def main():

    print("[1] IRN ...")
    df_irn = load_irn(IRN_ROOT)

    print("[2] SVM ...")
    df_svm = load_curve(SVM_DIR, "SVM_selected")

    print("[3] NCR ...")
    df_ncr = load_curve(NCR_DIR, "NCR_selected")

    print("[4] CELL ...")
    df_cell = load_cell(CELL_FILE)

    df = pd.concat([df_irn, df_svm, df_ncr, df_cell], ignore_index=True)

    df["mean"] = pd.to_numeric(df["mean"], errors="coerce")
    df["std"] = pd.to_numeric(df["std"], errors="coerce")

    df.to_csv(OUT_RAW, index=False)
    print("[INFO] saved:", OUT_RAW)


    # ======================================================
    # 5. FINAL STATISTICS (ROBUST)
    # ======================================================

    def safe_std(x):
        x = x.dropna()
        return np.std(x, ddof=1) if len(x) > 1 else np.nan

    def safe_sem(x):
        x = x.dropna()
        return np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

    stats = (
        df.groupby(["selection", "endpoint"])
        .apply(lambda g: pd.Series({
            "mean_cindex": g["mean"].mean(),

            # ✔ unify std across sources
            "std_cindex": np.nanmean(g["std"]),

            "sem_cindex": safe_sem(g["mean"]),

            "n": len(g)
        }))
        .reset_index()
    )

    stats.to_csv(OUT_STATS, index=False)
    print("[INFO] saved:", OUT_STATS)


if __name__ == "__main__":
    main()

### Supports Extended Data Figure 2d — Extract training-run files

Extracts the local training-run archive used by the protein-panel comparison; this cell does not draw a panel.


In [ ]:
!tar -xvf code_9_15_save.tar.gz

### Extended Data Figure 2d — Protein-panel performance

Draws the performance comparison across protein-selection strategies and exports its table and legend.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch


# ======================================================
# 0. Matplotlib settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# 1. CONFIG
# ======================================================

ENDPOINTS = ["CHD", "istroke", "ICVD"]

IRN_ROOT = Path("code_9_15_save/SingleModal_GS_Run_IRN")

SVM_DIR = Path("CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1322/curves")
NCR_DIR = Path("CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1324/curves")

CELL_FILE = "STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv"

OUTDIR = "figs2_final"
OUT_TABLE = os.path.join(OUTDIR, "FINAL_SELECTION_TABLE.csv")
OUT_FIG = os.path.join(OUTDIR, "FINAL_SELECTION_BARPLOT.pdf")
OUT_LEGEND = os.path.join(OUTDIR, "FINAL_SELECTION_BARPLOT_legend.pdf")

# 120 mm × 50 mm
FIG_W_MM = 120
FIG_H_MM = 50
MM_TO_INCH = 1 / 25.4
FIGSIZE = (FIG_W_MM * MM_TO_INCH, FIG_H_MM * MM_TO_INCH)

# endpoint display
ENDPOINT_LABELS = {
    "CHD": "CHD",
    "stroke": "Stroke",
    "istroke": "Ischemic stroke",
    "ICVD": "ASCVD"
}

# plot order
ORDER_SEL = ["IRN_all", "SVM_selected", "NCR_selected", "CELL_selected"]
ORDER_EP = ["CHD","istroke", "ICVD"]

# colors
COLORS = {
    "IRN_all": "#B7B3AE",        # warm morandi gray
    "SVM_selected": "#9E9B97",   # darker morandi gray
    "NCR_selected": "#C9C5C1",   # lighter morandi gray
    "CELL_selected": "#8B1E1E"   # deep red
}

# professional legend labels
LEGEND_LABELS = {
    "IRN_all": "All proteins (IRN normalization)",
    "SVM_selected": "SVM-selected proteins",
    "NCR_selected": "NCR-selected proteins (Schuermans et al., Nat Cardiovasc Res 2024)",
    "CELL_selected": "Cell-selected proteins (Deng et al., Cell 2025)"
}


# ======================================================
# 2. IRN (folder-level mean/std)
# ======================================================

def load_irn(root: Path):

    rows = []

    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        if not folder.name.startswith("soma"):
            continue

        f = folder / "summary.csv"
        if not f.exists():
            continue

        df = pd.read_csv(f)

        for _, r in df.iterrows():
            if r["endpoint"] not in ENDPOINTS:
                continue

            rows.append({
                "selection": "IRN_all",
                "endpoint": r["endpoint"],
                "value": float(r["cindex"])
            })

    df = pd.DataFrame(rows)

    out = (
        df.groupby(["selection", "endpoint"])
        .agg(
            mean=("value", "mean"),
            std=("value", "std")
        )
        .reset_index()
    )

    return out


# ======================================================
# 3. SVM / NCR (fold-level max -> mean/std)
# ======================================================

def load_curve(curve_dir: Path, name: str):

    rows = []
    pattern = re.compile(r"(CHD|stroke|istroke|ICVD)_.*fold(\d+).*\.csv")

    for f in curve_dir.glob("*_train_curve.csv"):

        m = pattern.search(f.name)
        if not m:
            continue

        endpoint = m.group(1)

        if endpoint not in ENDPOINTS:
            continue

        df = pd.read_csv(f)

        if "valid_ci" not in df.columns:
            continue

        rows.append({
            "selection": name,
            "endpoint": endpoint,
            "value": float(df["valid_ci"].max())
        })

    df = pd.DataFrame(rows)

    out = (
        df.groupby(["selection", "endpoint"])
        .agg(
            mean=("value", "mean"),
            std=("value", "std")
        )
        .reset_index()
    )

    return out


# ======================================================
# 4. CELL SELECTED (direct CV result)
# ======================================================

def load_cell(cell_file: str):

    df = pd.read_csv(cell_file)

    df = df[df["modality"] == "soma"].copy()
    df = df[df["model_tag"] == "MLP"].copy()
    df = df[df["endpoint"].isin(ENDPOINTS)].copy()

    df = df.rename(columns={
        "mean_valid_ci": "mean",
        "std_valid_ci": "std"
    })

    out = df[["endpoint", "mean", "std"]].copy()
    out["selection"] = "CELL_selected"

    return out[["selection", "endpoint", "mean", "std"]]


# ======================================================
# 5. Save standalone legend
# ======================================================

def save_separate_legend(out_path):

    handles = [
        Patch(facecolor=COLORS[sel], edgecolor="black", linewidth=0.5, label=LEGEND_LABELS[sel])
        for sel in ORDER_SEL
    ]

    fig_leg = plt.figure(figsize=(7.5, 1.5))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis("off")

    ax_leg.legend(
        handles=handles,
        loc="center",
        frameon=False,
        ncol=1,
        fontsize=8.5,
        handlelength=1.6,
        handleheight=0.9,
        columnspacing=1.2,
        labelspacing=0.8
    )

    plt.tight_layout()
    fig_leg.savefig(out_path, bbox_inches="tight", transparent=True)
    plt.close(fig_leg)


# ======================================================
# 6. Main pipeline
# ======================================================

def main():

    os.makedirs(OUTDIR, exist_ok=True)

    print("[1] IRN ...")
    df_irn = load_irn(IRN_ROOT)

    print("[2] SVM ...")
    df_svm = load_curve(SVM_DIR, "SVM_selected")

    print("[3] NCR ...")
    df_ncr = load_curve(NCR_DIR, "NCR_selected")

    print("[4] CELL ...")
    df_cell = load_cell(CELL_FILE)

    # --------------------------------------------------
    # Merge final table
    # --------------------------------------------------
    df = pd.concat([df_irn, df_svm, df_ncr, df_cell], ignore_index=True)

    df["selection"] = pd.Categorical(df["selection"], categories=ORDER_SEL, ordered=True)
    df["endpoint"] = pd.Categorical(df["endpoint"], categories=ORDER_EP, ordered=True)

    df = df.sort_values(["endpoint", "selection"]).reset_index(drop=True)

    df.to_csv(OUT_TABLE, index=False)
    print("[INFO] saved table:", OUT_TABLE)

    # --------------------------------------------------
    # Plot barplot only
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=FIGSIZE)

    x = np.arange(len(ORDER_EP))
    width = 0.18

    for i, sel in enumerate(ORDER_SEL):

        sub = (
            df[df["selection"] == sel]
            .set_index("endpoint")
            .reindex(ORDER_EP)
        )

        means = sub["mean"].values.astype(float)
        stds = sub["std"].values.astype(float)
        stds = np.nan_to_num(stds, nan=0.0)

        ax.bar(
            x + i * width,
            means,
            width=width,
            yerr=stds,
            capsize=2.5,
            color=COLORS[sel],
            edgecolor="black",
            linewidth=0.45,
            label=LEGEND_LABELS[sel],
            zorder=3,
            error_kw=dict(
                lw=0.8,
                capthick=0.8
            )
        )

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([ENDPOINT_LABELS[e] for e in ORDER_EP], rotation=0)

    ax.set_ylabel("C-index")
    ax.set_xlabel("")

    # no title
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)

    # main figure without legend
    # adjust y-range slightly
    y_min = 0.45
    y_max = max(0.72, np.nanmax(df["mean"] + df["std"]) + 0.02)
    ax.set_ylim(y_min, y_max)

    plt.tight_layout()
    fig.savefig(OUT_FIG, dpi=300, bbox_inches="tight", transparent=True)
    plt.close(fig)
    print("[INFO] saved figure:", OUT_FIG)

    # --------------------------------------------------
    # Save separate legend
    # --------------------------------------------------
    save_separate_legend(OUT_LEGEND)
    print("[INFO] saved legend:", OUT_LEGEND)


if __name__ == "__main__":
    main()

### Supporting analysis for Extended Data Figure 2d — Protein-panel GO enrichment

Runs GO enrichment for the compared protein panels. These GO plots have no separate panel number in the figure legends.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp


# ======================================================
# CONFIG
# ======================================================

SVM_FILE = "CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1322/explanations/CHD_soma_TR_best_IG.csv"
NCR_FILE = "CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1324/explanations/CHD_soma_TR_best_IG.csv"
CELL_FILE = "code_02_02_save/soma_20260202_2006/explanations/stroke_soma_TR_best_IG.csv"

OUT_DIR = "GO_enrichment_final"
os.makedirs(OUT_DIR, exist_ok=True)


# ======================================================
# 1. protein parsing (IMPORTANT FIX)
# ======================================================

def parse_protein(x):
    """
    seq.F12 -> F12
    """
    if pd.isna(x):
        return None

    x = str(x)

    if "." in x:
        x = x.split(".")[1]

    return x


def load_protein_list(path):

    df = pd.read_csv(path)

    if "feature" not in df.columns:
        raise ValueError(f"{path} missing feature column")

    genes = df["feature"].dropna().map(parse_protein)

    genes = genes.dropna().unique().tolist()

    return genes


# ======================================================
# 2. GO enrichment
# ======================================================

def run_go_enrichr(gene_list, out_prefix):

    gene_list = list(set(gene_list))

    print(f"\n[GO] Running: {out_prefix}")
    print(f"[GO] Genes: {len(gene_list)}")
    print(f"[GO] Example: {gene_list[:10]}")

    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=[
            "GO_Biological_Process_2023",
            "GO_Molecular_Function_2023",
            "GO_Cellular_Component_2023"
        ],
        organism="Human",
        outdir=out_prefix,
        cutoff=0.5,
        no_plot=True
    )

    res = enr.results.copy()

    res.to_csv(f"{out_prefix}_GO_full.csv", index=False)

    return res


# ======================================================
# 3. plot
# ======================================================

def plot_go(df, title, out_png):

    if df is None or len(df) == 0:
        print(f"[Plot] empty: {title}")
        return

    sig = df[df["Adjusted P-value"] < 0.05].copy()

    if len(sig) == 0:
        print(f"[Plot] no significant GO: {title}")
        return

    sig = sig.sort_values("Adjusted P-value").head(15)

    sig["score"] = -np.log10(sig["Adjusted P-value"] + 1e-300)

    plt.figure(figsize=(6, 5))

    plt.barh(sig["Term"][::-1], sig["score"][::-1])

    plt.xlabel("-log10 FDR")
    plt.title(title)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight", transparent=True)
    plt.show()


# ======================================================
# 4. pipeline
# ======================================================

def run_set(name, path):

    proteins = load_protein_list(path)

    out_prefix = os.path.join(OUT_DIR, name)

    res = run_go_enrichr(proteins, out_prefix)

    plot_go(
        res,
        f"{name} GO enrichment",
        os.path.join(OUT_DIR, f"{name}_GO.png")
    )

    return res


# ======================================================
# MAIN
# ======================================================

def main():

    print("\n==============================")
    print("GO enrichment pipeline")
    print("==============================\n")

    run_set("SVM_select", SVM_FILE)
    run_set("NCR_select", NCR_FILE)
    run_set("cell_select", CELL_FILE)

    print("\nDONE")
    print("Output:", OUT_DIR)


if __name__ == "__main__":
    main()

### Supporting analysis for Extended Data Figure 2d — Protein-panel overlap

Draws overlap ratios between the selected and reference protein panels. This overlap plot has no separate panel number in the figure legends.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import matplotlib.pyplot as plt


# ======================================================
# INPUT FILES
# ======================================================

SVM_FILE = "CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1322/explanations/CHD_soma_TR_best_IG.csv"
NCR_FILE = "CIPHER_save_11_18/SingleModal_Run_update_v2/soma_20251117_1324/explanations/CHD_soma_TR_best_IG.csv"
CELL_FILE = "code_02_02_save/soma_20260202_2006/explanations/stroke_soma_TR_best_IG.csv"


# ======================================================
# reference causal gene set
# ======================================================

CAUSAL_GENES = [
    "PCSK9","NOS3","LDLR","LPL","LIPA","ANGPTL4","CDK8","APOB","FN1",
    "CCM2","HNF1A","PNPLA3","REST","SCARB1","APOE","ABCG8","SEMA3C",
    "COL4A1","MECOM","SGCD","ACVRL1","RRP1B","SWAP7","IL6R","NME7",
    "LAMB2","ARHGEF26","EDNRA","GUCY1A3","LOX","PHACTR1","TCF21",
    "PLG","TBX20","ARHGAP42","APOC3","LIPC","MYH11","CETP","SREBF1",
    "CXCR4","MC4R","TGFB1","FES"
]


# ======================================================
# protein parsing
# ======================================================

def parse_protein(x):
    if pd.isna(x):
        return None
    x = str(x)
    if "." in x:
        x = x.split(".")[1]
    return x


def load_set(path):
    df = pd.read_csv(path)
    genes = df["feature"].dropna().map(parse_protein)
    genes = set([g for g in genes if g is not None])
    return genes


# ======================================================
# compute ratio only
# ======================================================

def compute_ratio(selected, reference):
    overlap = selected.intersection(reference)

    ratio = len(overlap) / len(selected) if len(selected) > 0 else 0

    return ratio


# ======================================================
# main
# ======================================================

def main():

    svm = load_set(SVM_FILE)
    ncr = load_set(NCR_FILE)
    cell = load_set(CELL_FILE)

    ref = set(CAUSAL_GENES)

    ratios = {
        "SVM_select": compute_ratio(svm, ref),
        "NCR_select": compute_ratio(ncr, ref),
        "Cell_select": compute_ratio(cell, ref),
    }

    print("\nOverlap ratio results:")
    for k, v in ratios.items():
        print(k, round(v, 4))

    # ==================================================
    # plot (ONLY ratio barplot)
    # ==================================================

    labels = list(ratios.keys())
    values = list(ratios.values())

    plt.figure(figsize=(5.5, 4))

    bars = plt.bar(labels, values)

    plt.ylabel("Causal gene overlap ratio")
    plt.title("Protein selection enrichment")

    # annotate ratio
    for i, v in enumerate(values):
        plt.text(i, v + 0.005, f"{v:.3f}", ha="center")

    plt.ylim(0, max(values) * 1.3 if max(values) > 0 else 1)

    plt.tight_layout()

    out = "gene_overlap_ratio_barplot.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()

    print("\nSaved:", out)


if __name__ == "__main__":
    main()